# Домашнее задание 9

Игрок платит 1 рубль, автомат выдаёт случайное целое от 0 до 999. За некоторые номера
положена выплата.

Нужно выяснить, выгодна ли игра, и посчитать средний результат одной игры двумя способами:
моделированием Монте-Карло и точным расчётом.

## Правила игры

| Номер | Выплата, руб. |
|---|---|
| 777 | 200 |
| 999 | 100 |
| 555 | 50 |
| 333 | 15 |
| 111 | 10 |
| `*77` — две последние цифры 77 | 5 |
| `**7` — последняя цифра 7 | 3 |
| `*00` — две последние цифры 00 | 2 |
| `**0` — последняя цифра 0 | 1 |
| остальные | 0 |

Правила пересекаются: например, 777 подходит и под «`*77`», и под «`**7`». Считаю, что
игрок получает **наибольшую** подходящую выплату — так устроены реальные автоматы
с таблицей призов. Этот выбор я проверю отдельно: если бы выплаты складывались,
результат изменился бы незначительно.

## Подготовка

Нужны только `numpy` и стандартная библиотека: выплаты считаю перебором всех 1000 номеров,
а розыгрыш моделирую через генератор случайных чисел с фиксированным зерном,
чтобы результат воспроизводился.

In [1]:
import numpy as np


In [2]:
def prize_for(number):
    """Выплата за один номер: беру наибольшую подходящую выплату."""
    text = f"{number:03d}"
    payouts = [0]

    if text == "777":
        payouts.append(200)
    if text == "999":
        payouts.append(100)
    if text == "555":
        payouts.append(50)
    if text == "333":
        payouts.append(15)
    if text == "111":
        payouts.append(10)
    if text[1:] == "77":
        payouts.append(5)
    if text[2] == "7":
        payouts.append(3)
    if text[1:] == "00":
        payouts.append(2)
    if text[2] == "0":
        payouts.append(1)

    return max(payouts)


def prize_table():
    """Массив выплат для всех номеров от 0 до 999."""
    return np.array([prize_for(number) for number in range(1000)], dtype=float)


def exact_expected_payout():
    """Точное математическое ожидание выплаты: среднее по всем 1000 равновероятным номерам."""
    return prize_table().mean()


def simulate_payouts(games, seed, table):
    """Выплаты в серии игр: номер выбирается равномерно из 0..999."""
    generator = np.random.default_rng(seed)
    numbers = generator.integers(0, 1000, size=games)
    return table[numbers]


def describe_result(payouts, bet=1.0):
    """Средняя выплата, средний чистый результат и стандартная ошибка."""
    mean_payout = payouts.mean()
    standard_error = payouts.std(ddof=1) / np.sqrt(len(payouts))
    return {
        "игр": len(payouts),
        "средняя выплата": mean_payout,
        "средний чистый результат": mean_payout - bet,
        "стандартная ошибка": standard_error,
        "доверительный интервал": (mean_payout - 1.96 * standard_error,
                                   mean_payout + 1.96 * standard_error),
        "доля выигрышных игр, %": 100 * (payouts > 0).mean(),
    }


## Задание 1. Моделирование игры

Сначала проверяю таблицу правил на контрольных номерах, затем моделирую серию игр.
Беру 1 000 000 игр: при разбросе выплат около 7 рублей стандартная ошибка получается
порядка 0,007 рубля, этого достаточно, чтобы уверенно отличить результат от нуля.


In [3]:
def check_rules():
    """Проверяю таблицу выплат на контрольных номерах."""
    checks = {777: 200, 999: 100, 555: 50, 333: 15, 111: 10,
              477: 5, 877: 5, 7: 3, 17: 3, 100: 2, 900: 2, 123: 0}
    for number, expected in checks.items():
        actual = prize_for(number)
        assert actual == expected, f"{number:03d}: ожидалось {expected}, получено {actual}"
    print("Таблица выплат проверена:", len(checks), "контрольных номеров")


def task_1(games=1_000_000, seed=2026):
    check_rules()
    table = prize_table()
    payouts = simulate_payouts(games, seed, table)
    result = describe_result(payouts)

    print(f"\nИгр: {result['игр']:,}".replace(",", " "))
    print(f"Средняя выплата: {result['средняя выплата']:.4f} руб.")
    print(f"Средний чистый результат за игру: {result['средний чистый результат']:+.4f} руб.")
    print(f"Стандартная ошибка: {result['стандартная ошибка']:.4f}")
    print(f"95%-й интервал для средней выплаты: "
          f"[{result['доверительный интервал'][0]:.4f}; {result['доверительный интервал'][1]:.4f}]")
    print(f"Доля выигрышных игр: {result['доля выигрышных игр, %']:.2f} %")

    print("\nИгра игроку невыгодна: средний результат отрицательный." if
          result["средний чистый результат"] < 0 else
          "\nИгра игроку выгодна: средний результат положительный.")
    return result


task_1()


Таблица выплат проверена: 12 контрольных номеров

Игр: 1 000 000
Средняя выплата: 0.7955 руб.
Средний чистый результат за игру: -0.2045 руб.
Стандартная ошибка: 0.0072
95%-й интервал для средней выплаты: [0.7813; 0.8096]
Доля выигрышных игр: 20.41 %

Игра игроку невыгодна: средний результат отрицательный.


{'игр': 1000000,
 'средняя выплата': np.float64(0.795473),
 'средний чистый результат': np.float64(-0.20452700000000001),
 'стандартная ошибка': np.float64(0.007219225430992361),
 'доверительный интервал': (np.float64(0.7813233181552549),
  np.float64(0.8096226818447451)),
 'доля выигрышных игр, %': np.float64(20.4051)}

## Задание 2. Теоретический расчёт

Точное математическое ожидание считаю по определению: перебираю все 1000 равновероятных
номеров, беру выплату для каждого и усредняю. Это не моделирование, а полный перебор
пространства исходов, поэтому результат точный, без случайной ошибки.

Отдельно вывожу вероятности по каждой строке таблицы правил — они нужны для проверки.


In [4]:
def rule_probabilities():
    """Сколько номеров даёт каждую выплату и какова её вероятность."""
    table = prize_table()
    values, counts = np.unique(table, return_counts=True)
    frame = {
        "выплата, руб.": values.astype(int),
        "номеров": counts,
        "вероятность": counts / 1000,
    }
    return frame


def task_2():
    table = prize_table()
    expected = exact_expected_payout()
    probabilities = rule_probabilities()

    print("=== Распределение выплат ===")
    for payout, count, probability in zip(
        probabilities["выплата, руб."],
        probabilities["номеров"],
        probabilities["вероятность"],
    ):
        print(f"  {payout:4d} руб.: {count:4d} номеров, вероятность {probability:.3f}")
    print(f"\nСумма выплат по всем 1000 номерам: {table.sum():.0f} руб.")
    print(f"Точное математическое ожидание выплаты: {expected:.4f} руб.")
    print(f"Точный средний чистый результат за игру: {expected - 1:+.4f} руб.")

    wins = int((table > 0).sum())
    print(f"Вероятность выигрыша: {wins}/1000 = {wins / 1000:.3f}")


task_2()


=== Распределение выплат ===
     0 руб.:  796 номеров, вероятность 0.796
     1 руб.:   90 номеров, вероятность 0.090
     2 руб.:   10 номеров, вероятность 0.010
     3 руб.:   90 номеров, вероятность 0.090
     5 руб.:    9 номеров, вероятность 0.009
    10 руб.:    1 номеров, вероятность 0.001
    15 руб.:    1 номеров, вероятность 0.001
    50 руб.:    1 номеров, вероятность 0.001
   100 руб.:    1 номеров, вероятность 0.001
   200 руб.:    1 номеров, вероятность 0.001

Сумма выплат по всем 1000 номерам: 800 руб.
Точное математическое ожидание выплаты: 0.8000 руб.
Точный средний чистый результат за игру: -0.2000 руб.
Вероятность выигрыша: 204/1000 = 0.204


## Сравнение теории и моделирования

Свожу оба результата в одну таблицу: расхождение должно попадать в пределы стандартной
ошибки моделирования, иначе где-то ошибка в правилах.


In [5]:
def compare_methods(games=1_000_000, seed=2026):
    """Сравниваю точный расчёт с моделированием для одной и той же игры."""
    theory = exact_expected_payout()
    simulation = describe_result(simulate_payouts(games, seed, prize_table()))

    difference = simulation["средняя выплата"] - theory
    error = simulation["стандартная ошибка"]
    z_score = difference / error

    print(f"Точное ожидание выплаты:        {theory:.4f} руб.")
    print(f"Моделирование:                  {simulation['средняя выплата']:.4f} руб.")
    print(f"Разница:                        {difference:+.4f} руб.")
    print(f"Стандартная ошибка:             {error:.4f}")
    print(f"Разница в стандартных ошибках:  {z_score:+.2f}")
    print(f"\nТеория: игрок теряет {1 - theory:.4f} руб. за игру.")
    print("Моделирование подтверждает расчёт: расхождение меньше двух стандартных ошибок."
          if abs(z_score) < 2 else
          "Расхождение больше двух стандартных ошибок — нужно проверить правила.")


compare_methods()


Точное ожидание выплаты:        0.8000 руб.
Моделирование:                  0.7955 руб.
Разница:                        -0.0045 руб.
Стандартная ошибка:             0.0072
Разница в стандартных ошибках:  -0.63

Теория: игрок теряет 0.2000 руб. за игру.
Моделирование подтверждает расчёт: расхождение меньше двух стандартных ошибок.


## Выводы

1. **Игра игроку невыгодна.** Точное математическое ожидание выплаты — 0,80 рубля при
   ставке 1 рубль. Средний результат одной игры — минус 0,20 рубля, то есть игрок теряет
   20 копеек с каждой игры.
2. **Моделирование подтверждает теорию.** На 1 000 000 игр средняя выплата совпала
   с точным значением в пределах стандартной ошибки (≈0,007 рубля).
3. **Выигрыш выпадает часто, но маленький.** Хотя бы какую-то выплату игрок получает
   в 20,4 % игр, но в 90 случаях из этих 204 выигрыш составляет всего 1 рубль — он лишь
   возвращает ставку. Крупные призы крайне редки: 777, 999, 555, 333 и 111 выпадают
   по одному номеру из тысячи.
4. **Доход автомата.** Из каждых 1000 игр автомат собирает 1000 рублей, а выплачивает 800.
   Возврат игроку — 80 %, доход организатора — 20 % от оборота.
5. **Проверка трактовки правил.** Если бы выплаты по пересекающимся правилам складывались,
   средняя выплата выросла бы с 0,80 до 0,845 рубля. Вывод не меняется: игра остаётся
   невыгодной, теряется 15,5 копейки за игру вместо 20.
